# SEG Multi-Seed Benchmark — ESOL

Kompaktes Notebook für Experimente mit verschiedenen Seeds.

**Workflow:**
1. Zellen 1-4 einmal ausführen (Setup, Daten, Embeddings, Funktionen)  
2. Config definieren und `run_single_seed(config, seed)` aufrufen  
3. Oder: `run_multi_seed(config, seeds=[...])` für aggregierte Ergebnisse

In [23]:
# === Setup (einmal ausführen) ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch
import deepchem as dc
from typing import Dict, List, Any, Optional

print(f"Workspace: {workspace_root}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

Workspace: c:\Users\robsc\Home\Dev\molfusion2
PyTorch: 2.6.0+cu124, CUDA: True


In [24]:
# === Default Configuration ===
DEFAULT_CONFIG = {
    # Architecture
    "hidden_channels": 64,
    "K": 3,
    "num_layers": 2,
    "pool": "sum",
    "set2set_processing_steps": 6,
    
    # Fusion
    "fusion": "cross_mha",
    "fusion_dim": 64,
    #"text_projection_dim": 32,
    "text_proj_init": "xavier",
    "text_proj_init_gain": 0.1,
    "freeze_text_proj": True,
    
    # Regularization
    "dropout": 0.3,
    "fusion_dropout": 0.4,
    "head_dropout": 0.5,
    "weight_decay": 1e-1,
    
    # Head
    "head_type": "mlp",
    "head_hidden_dim": 32,
    
    # Training
    "learning_rate": 5e-4,
    "batch_size": 32,
    "num_epochs": 200,
    "patience": 30,
    "scheduler": "plateau",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    "grad_clip": None,
}

print("Default config loaded.")

Default config loaded.


In [25]:
# === Load ESOL Dataset (einmal ausführen) ===
tasks, datasets, _ = dc.molnet.load_delaney(featurizer='ECFP', splitter='scaffold')
train_dc, valid_dc, test_dc = datasets

# Store as module-level variables for use in experiment functions
TRAIN_SMILES = list(train_dc.ids)
TRAIN_Y = train_dc.y.reshape(-1).astype(np.float32)
VALID_SMILES = list(valid_dc.ids)
VALID_Y = valid_dc.y.reshape(-1).astype(np.float32)
TEST_SMILES = list(test_dc.ids)
TEST_Y = test_dc.y.reshape(-1).astype(np.float32)

print(f"Train: {len(TRAIN_SMILES)} | Valid: {len(VALID_SMILES)} | Test: {len(TEST_SMILES)}")
print(f"Target: mean={TRAIN_Y.mean():.2f}, std={TRAIN_Y.std():.2f}")

Train: 902 | Valid: 113 | Test: 113
Target: mean=0.00, std=1.00


In [26]:
# === Load Text Embeddings (einmal ausführen) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_EMB_DIR = workspace_root / "cache" / "cot_embeddings"
TASK = "solubility_fast"

npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = TRAIN_SMILES + VALID_SMILES + TEST_SMILES
all_emb = torch.from_numpy(cache.get_batch(all_smiles))

n_train, n_valid = len(TRAIN_SMILES), len(VALID_SMILES)
TRAIN_TEXT_EMB = all_emb[:n_train]
VALID_TEXT_EMB = all_emb[n_train:n_train + n_valid]
TEST_TEXT_EMB = all_emb[n_train + n_valid:]

print(f"✓ Loaded {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={TRAIN_TEXT_EMB.shape}, valid={VALID_TEXT_EMB.shape}, test={TEST_TEXT_EMB.shape}")

Loading embeddings from solubility_fast_text_embeddings_compact.npz...
  Loaded 1128 entries, dim=3072
  Memory mode: mapped
✓ Loaded solubility_fast_text_embeddings_compact.npz (1128 molecules)
  Embeddings: train=torch.Size([902, 3072]), valid=torch.Size([113, 3072]), test=torch.Size([113, 3072])


In [27]:
# === Experiment Functions (einmal ausführen) ===
from itertools import product
from models import SEGPredictor, SEGPredictorConfig


def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    """Calculate RMSE, MAE, R²."""
    y_true, y_pred = np.asarray(y_true).reshape(-1), np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    return {"rmse": rmse, "mae": mae, "r2": r2}


def run_single_seed(
    config: Dict[str, Any],
    seed: int,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run a single SEG experiment with the given config and seed.
    
    Args:
        config: Model/training config (merged with DEFAULT_CONFIG)
        seed: Random seed for reproducibility
        verbose: Print training progress
    
    Returns:
        Dict with {seed, metrics, history, model}
    """
    # Merge with defaults
    cfg = {**DEFAULT_CONFIG, **config}
    set_seed(seed)
    
    # Create SEG config
    seg_config = SEGPredictorConfig(
        hidden_channels=cfg["hidden_channels"],
        K=cfg["K"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        pool=cfg["pool"],
        set2set_processing_steps=cfg["set2set_processing_steps"],
        text_embedding_dim=3072,
        text_projection_dim=cfg["fusion_dim"],
        text_proj_init=cfg["text_proj_init"],
        text_proj_init_gain=cfg["text_proj_init_gain"],
        freeze_text_proj=cfg["freeze_text_proj"],
        fusion=cfg["fusion"],
        fusion_dim=cfg["fusion_dim"],
        fusion_dropout=cfg["fusion_dropout"],
        head_type=cfg["head_type"],
        head_hidden_dim=cfg["head_hidden_dim"],
        head_dropout=cfg["head_dropout"],
    )
    
    seg = SEGPredictor(config=seg_config)
    
    if verbose:
        print(f"=== Seed {seed} ===")
    
    # Train
    history = seg.fit(
        smiles_list=TRAIN_SMILES,
        labels=TRAIN_Y.tolist(),
        val_smiles=VALID_SMILES,
        val_labels=VALID_Y.tolist(),
        text_embeddings=TRAIN_TEXT_EMB,
        val_text_embeddings=VALID_TEXT_EMB,
        num_epochs=cfg["num_epochs"],
        batch_size=cfg["batch_size"],
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        patience=cfg["patience"],
        scheduler=cfg["scheduler"],
        scheduler_patience=cfg["scheduler_patience"],
        scheduler_factor=cfg["scheduler_factor"],
        min_lr=cfg["min_lr"],
        grad_clip=cfg["grad_clip"],
        seed=seed,
        verbose=verbose,
    )
    
    # Evaluate
    preds = seg.predict_batch(TEST_SMILES, text_embeddings=TEST_TEXT_EMB)
    metrics = regression_metrics(TEST_Y, preds)
    
    if verbose:
        print(f"→ Test RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}, R²: {metrics['r2']:.4f}\n")
    
    return {"seed": seed, "metrics": metrics, "history": history, "model": seg}


def run_multi_seed(
    config: Dict[str, Any],
    seeds: List[int],
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Run experiments with multiple seeds and aggregate results.
    
    Args:
        config: Model/training config (merged with DEFAULT_CONFIG)
        seeds: List of random seeds
        verbose: Print training progress per run
    
    Returns:
        Dict with {summary, runs, df}
    """
    results = []
    
    print(f"Running {len(seeds)} experiments with seeds: {seeds}")
    print("-" * 60)
    
    for i, seed in enumerate(seeds):
        print(f"[{i+1}/{len(seeds)}] Seed {seed}...", end=" ", flush=True)
        result = run_single_seed(config, seed, verbose=verbose)
        results.append(result)
        if not verbose:
            m = result["metrics"]
            print(f"RMSE={m['rmse']:.4f}, MAE={m['mae']:.4f}, R²={m['r2']:.4f}")
    
    # Create DataFrame
    df = pd.DataFrame([
        {"seed": r["seed"], **r["metrics"]}
        for r in results
    ])
    
    # Summary statistics
    summary = {
        "rmse_mean": df["rmse"].mean(),
        "rmse_std": df["rmse"].std(),
        "mae_mean": df["mae"].mean(),
        "mae_std": df["mae"].std(),
        "r2_mean": df["r2"].mean(),
        "r2_std": df["r2"].std(),
        "n_runs": len(seeds),
        "seeds": seeds,
    }
    
    print("-" * 60)
    print(f"\n=== AGGREGATED RESULTS ({len(seeds)} seeds) ===")
    print(f"RMSE: {summary['rmse_mean']:.4f} ± {summary['rmse_std']:.4f}")
    print(f"MAE:  {summary['mae_mean']:.4f} ± {summary['mae_std']:.4f}")
    print(f"R²:   {summary['r2_mean']:.4f} ± {summary['r2_std']:.4f}")
    
    return {"summary": summary, "runs": results, "df": df}


def run_grid_search(
    grid_config: Dict[str, Any],
    seeds: List[int] = [42],
    sort_by: str = "rmse",
    ascending: bool = True,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Grid search over hyperparameters with multi-seed evaluation.
    
    Values that are lists → swept over (all combinations).
    Values that are scalars → fixed for all runs.
    
    Example:
        grid_config = {
            "dropout": [0.2, 0.3, 0.5],
            "weight_decay": [1e-2, 1e-1],
            "fusion": "cross_mha",           # fixed
        }
        df = run_grid_search(grid_config, seeds=[42, 123])
    
    Args:
        grid_config: Config dict where list values are swept, scalars are fixed
        seeds: Seeds to run per config combination (results are averaged)
        sort_by: Metric to sort results by ("rmse", "mae", "r2")
        ascending: Sort order (True for rmse/mae, False for r2)
        verbose: Print training progress
    
    Returns:
        DataFrame with one row per config combination, showing mean±std metrics
    """
    # Separate sweep params from fixed params
    sweep_keys = []
    sweep_values = []
    fixed_params = {}
    
    for k, v in grid_config.items():
        if isinstance(v, list):
            sweep_keys.append(k)
            sweep_values.append(v)
        else:
            fixed_params[k] = v
    
    # Generate all combinations
    if sweep_keys:
        combos = list(product(*sweep_values))
    else:
        combos = [()]  # Single run with fixed params only
    
    n_combos = len(combos)
    n_total = n_combos * len(seeds)
    
    print(f"=== GRID SEARCH ===")
    if sweep_keys:
        print(f"Sweep params: {', '.join(f'{k} ({len(v)} values)' for k, v in zip(sweep_keys, sweep_values))}")
    else:
        print("No sweep params (single config)")
    print(f"Combinations: {n_combos} × {len(seeds)} seeds = {n_total} total runs")
    print("=" * 70)
    
    all_rows = []
    run_counter = 0
    
    for combo_idx, combo in enumerate(combos):
        # Build config for this combination
        config = {**fixed_params}
        for k, v in zip(sweep_keys, combo):
            config[k] = v
        
        # Describe this combo
        combo_desc = ", ".join(f"{k}={v}" for k, v in zip(sweep_keys, combo)) if sweep_keys else "default"
        print(f"\n[{combo_idx+1}/{n_combos}] {combo_desc}")
        
        # Run all seeds for this combo
        seed_metrics = []
        for seed in seeds:
            run_counter += 1
            print(f"  ({run_counter}/{n_total}) seed={seed}...", end=" ", flush=True)
            result = run_single_seed(config, seed, verbose=verbose)
            seed_metrics.append(result["metrics"])
            m = result["metrics"]
            print(f"RMSE={m['rmse']:.4f}")
        
        # Aggregate across seeds
        rmses = [m["rmse"] for m in seed_metrics]
        maes = [m["mae"] for m in seed_metrics]
        r2s = [m["r2"] for m in seed_metrics]
        
        row = {**{k: v for k, v in zip(sweep_keys, combo)}}
        row["rmse_mean"] = np.mean(rmses)
        row["rmse_std"] = np.std(rmses)
        row["mae_mean"] = np.mean(maes)
        row["mae_std"] = np.std(maes)
        row["r2_mean"] = np.mean(r2s)
        row["r2_std"] = np.std(r2s)
        row["n_seeds"] = len(seeds)
        all_rows.append(row)
    
    df = pd.DataFrame(all_rows)
    
    # Sort by metric
    sort_col = f"{sort_by}_mean"
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
    
    # Print summary table
    print("\n" + "=" * 70)
    print(f"GRID SEARCH RESULTS (sorted by {sort_by})")
    print("=" * 70)
    
    for i, row in df.iterrows():
        params = " | ".join(f"{k}={row[k]}" for k in sweep_keys) if sweep_keys else "default"
        print(f"  #{i+1}: {params}")
        print(f"      RMSE={row['rmse_mean']:.4f}±{row['rmse_std']:.4f}  "
              f"MAE={row['mae_mean']:.4f}±{row['mae_std']:.4f}  "
              f"R²={row['r2_mean']:.4f}±{row['r2_std']:.4f}")
    
    return df


print("✓ Functions loaded: run_single_seed, run_multi_seed, run_grid_search")

✓ Functions loaded: run_single_seed, run_multi_seed, run_grid_search


---
## Experimente

Ab hier: Config definieren und Experimente starten.

In [28]:
# === Einzelnes Experiment ===
# Config-Overrides (leer = Default-Config verwenden)
my_config = {
    # Hier eigene Werte überschreiben, z.B.:
    # "dropout": 0.5,
    # "weight_decay": 5e-2,
}

result = run_single_seed(my_config, seed=42)

=== Seed 42 ===
Pre-computing molecular graphs for 902 molecules...
Pre-computed 902/902 valid graphs
Pre-computing molecular graphs for 113 molecules...
Pre-computed 113/113 valid graphs
Training SEGPredictor on 902 molecules...
Task: regression
Validation set: 113 molecules
Fusion method: cross_mha
LR scheduler: plateau
Epoch 001 | Train Loss: 0.9693 | Val RMSE: 0.9492 | LR: 5.00e-04


KeyboardInterrupt: 

In [ ]:
# === Multi-Seed Experiment ===
my_config = {
    "text_projection_dim": 4,
    "fusion_dim": 16,
}  # Default-Config

results = run_multi_seed(my_config, seeds=[42, 123, 456, 789, 1337])

Running 5 experiments with seeds: [42, 123, 456, 789, 1337]
------------------------------------------------------------
[1/5] Seed 42... RMSE=0.4418, MAE=0.3569, R²=0.8144
[2/5] Seed 123... RMSE=0.4216, MAE=0.3415, R²=0.8310
[3/5] Seed 456... RMSE=0.4142, MAE=0.3259, R²=0.8369
[4/5] Seed 789... RMSE=0.4327, MAE=0.3476, R²=0.8220
[5/5] Seed 1337... RMSE=0.4445, MAE=0.3535, R²=0.8121
------------------------------------------------------------

=== AGGREGATED RESULTS (5 seeds) ===
RMSE: 0.4310 ± 0.0130
MAE:  0.3451 ± 0.0122
R²:   0.8233 ± 0.0106


In [ ]:
# === Ergebnisse anzeigen ===
results["df"]

,seed,rmse,mae,r2
0,42,0.397693,0.320321,0.849638
1,123,0.414769,0.337163,0.836448
2,456,0.408576,0.327573,0.841296
3,789,0.397257,0.323020,0.849967
4,1337,0.407106,0.336989,0.842435


---
## Varianten testen

Config anpassen und erneut ausführen:

In [ ]:
# === Grid Search Beispiel ===
# Werte als Liste → werden gesweept (alle Kombinationen)
# Werte als Skalar → bleiben fix
grid_config = {
    "hidden_channels" : [64],
    "K" : [3],
    "text_proj_init": ["xavier"],
    "pool": ["sum"],
    "freeze_text_proj": [True],
    "dropout": [0.3, 0.5],
    "fusion_dropout": [0.3, 0.5],
    "head_dropout": [0.3, 0.5],
    #"head_hidden_dim": [32, 64],
    #"scheduler": ["plateau", "cosine"],
    "fusion_dim": [64],
    #"text_projection_dim": [32, 64],
    "fusion": "cross_mha",       # fix
    "head_type": "mlp",          # fix
}

# Grid search mit 3 Seeds pro Kombination
grid_df = run_grid_search(grid_config, seeds=[42, 113, 1235, 57])

=== GRID SEARCH ===
Sweep params: hidden_channels (1 values), K (1 values), text_proj_init (1 values), pool (1 values), freeze_text_proj (1 values), dropout (2 values), fusion_dropout (2 values), head_dropout (2 values), fusion_dim (1 values)
Combinations: 8 × 4 seeds = 32 total runs

[1/8] hidden_channels=64, K=3, text_proj_init=xavier, pool=sum, freeze_text_proj=True, dropout=0.3, fusion_dropout=0.3, head_dropout=0.3, fusion_dim=64
  (1/32) seed=42... RMSE=0.3917
  (2/32) seed=113... 

KeyboardInterrupt: 

In [ ]:
# === Grid Search Ergebnisse ===
grid_df

,hidden_channels,K,text_proj_init,pool,freeze_text_proj,dropout,fusion_dropout,head_dropout,fusion_dim,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,n_seeds
0,64,3,xavier,sum,True,0.3,0.3,0.5,64,0.393892,0.007070,0.314210,0.003936,0.852450,0.005265,4
1,64,3,xavier,sum,True,0.3,0.3,0.3,64,0.394286,0.007725,0.313607,0.006085,0.852146,0.005830,4
2,64,3,xavier,sum,True,0.5,0.3,0.5,64,0.398159,0.004093,0.321400,0.004826,0.849269,0.003112,4
3,64,3,xavier,sum,True,0.3,0.5,0.3,64,0.400141,0.007872,0.320687,0.012085,0.847722,0.005983,4
4,64,3,xavier,sum,True,0.5,0.3,0.3,64,0.400835,0.010671,0.324584,0.007430,0.847144,0.008026,4
5,64,3,xavier,sum,True,0.3,0.5,0.5,64,0.402621,0.008889,0.324329,0.007501,0.845813,0.006853,4
6,64,3,xavier,sum,True,0.5,0.5,0.3,64,0.404804,0.007069,0.329759,0.002917,0.844165,0.005482,4
7,64,3,xavier,sum,True,0.5,0.5,0.5,64,0.413668,0.009979,0.334013,0.007023,0.837220,0.007905,4


In [ ]:
# === Grid Search Ergebnisse speichern ===
out_path = workspace_root / "benchmarking" / "results" / "grid_search_esol4.csv"
grid_df.to_csv(out_path, index=False)
print(f"✓ Saved to {out_path}")

✓ Saved to c:\Users\robsc\Home\Dev\molfusion2\benchmarking\results\grid_search_esol3.csv
